# Prompt Optimization & Redeploy

## Automatically improve your BGP banking agent's prompt with GEPA

Now that you've evaluated your agent, collected feedback, and aligned your judges, it's time to **automatically optimize** the system prompt. MLflow's GEPA (Genetic Prompt Adaptation) optimizer iterates on your prompt to maximize quality scores.

After optimization, you'll register the improved prompt and **redeploy your app** to see the difference.

## Install packages (only required if running in a Databricks Notebook)

In [ ]:
%pip install -U -r ../requirements.txt
dbutils.library.restartPython()

## Environment Setup

In [ ]:
import sys
sys.path.append('../')
sys.path.append('../../')

import os
import mlflow
from mlflow_demo.utils import *

# Configure for the current participant: derives the object prefix, prompt name,
# model service, and experiment from YOUR username.
setup_databricks_notebook_env()

print(f'Experiment ID: {os.getenv("MLFLOW_EXPERIMENT_ID")}')
print(f'UC Schema: {os.getenv("UC_CATALOG")}.{os.getenv("UC_SCHEMA")}')
print(f'Prompt: {os.getenv("PROMPT_NAME")}')

# Step 1: Load the Current Production Prompt

Load the prompt your agent is currently using from the UC Prompt Registry.

In [ ]:
UC_CATALOG = os.environ.get('UC_CATALOG')
UC_SCHEMA = os.environ.get('UC_SCHEMA')
PROMPT_NAME = os.environ.get('PROMPT_NAME', 'bgp_support_prompt')
PROMPT_ALIAS = os.environ.get('PROMPT_ALIAS', 'production')

prompt_full_name = f'{UC_CATALOG}.{UC_SCHEMA}.{PROMPT_NAME}'
prompt_uri = f'prompts:/{prompt_full_name}@{PROMPT_ALIAS}'

current_prompt = mlflow.genai.load_prompt(prompt_uri)
print(f'Loaded prompt: {prompt_full_name}@{PROMPT_ALIAS}')
print(f'Template length: {len(current_prompt.template)} chars')
print(f'Preview: {current_prompt.template[:200]}...')

# Step 2: Define Training Questions and Scorers

GEPA evaluates candidate prompts against a small set of training questions. We use 3 banking questions and the same scorers from the evaluation notebook.

In [ ]:
from mlflow.genai.optimize import GepaPromptOptimizer
from mlflow.genai.scorers import Guidelines
from mlflow_demo.agent.agent import AGENT
from mlflow.types.responses import Message, ResponsesAgentRequest

TRAIN_QUESTIONS = [
    'What is the balance on my checking account?',
    'Can you show my recent transactions from the last 30 days?',
    'What loans do I currently have and what are the outstanding balances?',
]

SCORER_GUIDELINES = {
    'accuracy': 'The response must only cite account numbers, balances, and transaction amounts that appear in tool call results. No fabricated financial data.',
    'relevance': 'The response must directly answer the banking question about accounts, transactions, or loans.',
    'actionability': 'The response must include specific next steps or guidance a banking customer could follow.',
}

scorers = [
    Guidelines(name=name, guidelines=text)
    for name, text in SCORER_GUIDELINES.items()
]

train_data = [
    {'inputs': {'input': [{'role': 'user', 'content': q}]}}
    for q in TRAIN_QUESTIONS
]

print(f'Training questions: {len(TRAIN_QUESTIONS)}')
print(f'Scorers: {list(SCORER_GUIDELINES.keys())}')

# Step 3: Run GEPA Optimization

This runs a lightweight optimization with `max_metric_calls=20` (vs the default 100). It should complete in 1-3 minutes.

In [ ]:
from datetime import datetime

mlflow.set_experiment(experiment_id=os.environ['MLFLOW_EXPERIMENT_ID'])

reflection_model = os.environ.get(
    'REFLECTION_MODEL', 'databricks:/databricks-claude-sonnet-4-6'
)

def predict_fn(input):
    request = ResponsesAgentRequest(input=input)
    return AGENT.predict(request)

run_name = f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_gepa_banking'

with mlflow.start_run(run_name=run_name) as run:
    result = mlflow.genai.optimize_prompts(
        predict_fn=predict_fn,
        train_data=train_data,
        prompt_uris=[current_prompt.uri],
        optimizer=GepaPromptOptimizer(
            reflection_model=reflection_model,
            max_metric_calls=20,
            display_progress_bar=False,
        ),
        scorers=scorers,
    )
    run_id = run.info.run_id

optimized = result.optimized_prompts[0]
initial_score = float(getattr(result, 'initial_eval_score', 0.0) or 0.0)
final_score = float(getattr(result, 'final_eval_score', 0.0) or 0.0)

print(f'Initial score: {initial_score:.3f}')
print(f'Final score:   {final_score:.3f}')
print(f'Improved: {final_score > initial_score}')
print(f'Run ID: {run_id}')
print(f'\nOptimized prompt preview:\n{optimized.template[:500]}...')

# Step 4: Register the Optimized Prompt

If the optimized prompt scores higher than the original, register it as a new version and promote it to production.

In [ ]:
improved = final_score > initial_score

if improved:
    new_version = mlflow.genai.register_prompt(
        name=prompt_full_name,
        template=optimized.template,
        commit_message=f'GEPA optimization (score {initial_score:.3f} -> {final_score:.3f})',
        tags={
            'optimization': 'GEPA-banking',
            'initial_score': f'{initial_score:.3f}',
            'final_score': f'{final_score:.3f}',
            'run_id': run_id,
        },
    )
    print(f'Registered new prompt version: {new_version.version}')

    mlflow.genai.set_prompt_alias(
        name=prompt_full_name,
        alias='production',
        version=new_version.version,
    )
    print(f'Set production alias to version {new_version.version}')
else:
    print(f'Score did not improve ({initial_score:.3f} -> {final_score:.3f}); skipping registration')

# Step 5: Redeploy Your App

Now that the production alias points to the optimized prompt, redeploy your app to pick up the new prompt.

Run this in your terminal (not in the notebook):
```bash
cd /path/to/mlflow-end-to-end-genai
./deploy.sh
```

Then visit your app URL and ask the same banking question you asked in Phase 2. Compare the response quality before and after optimization.

---
## Exercise

Ask the same banking question to your redeployed app that you asked before optimization. Can you see the improvement? Which quality dimension improved the most?

## Summary

You've completed the full MLflow GenAI quality loop:
1. **Observed** your agent with tracing
2. **Evaluated** with LLM judges and code-based scorers
3. **Found and fixed** quality issues by improving the prompt
4. **Collected** human feedback through labeling
5. **Monitored** quality in production
6. **Optimized** the prompt automatically with GEPA and redeployed

This is the same cycle you can apply to your own GenAI applications at BGP Bank!